# Семинар 05. Модель данных и контейнеры

На семинаре мы будем сначала предсказывать поведение программы, затем проверять гипотезу и исправлять модель данных.

## Цели

- находить ошибки, вызванные общими ссылками;
- отличать сравнение значений от проверки идентичности;
- выбирать подходящие контейнеры для прикладных данных;
- строить независимые копии вложенных структур;
- считать денежные величины без ошибки `float`.

## Перед началом

Откройте [лекцию](lecture.ipynb) и держите рядом таблицу встроенных контейнеров. В каждой задаче запишите ожидаемый результат до запуска ячейки. Код с `TODO` является заготовкой и может не проходить проверки до вашего решения.

## Разминка: значение или объект?

Не выполняя код, для каждой строки определите результаты `==` и `is`. Затем объясните их через объекты и ссылки, не используя фразу «Python решил скопировать переменную».

In [ ]:
first = [10, 20]
second = first
third = first.copy()

# Сначала запишите прогноз, затем снимите комментарий.
# print(first == second, first is second)
# print(first == third, first is third)
# second.append(30)
# print(first, third)

## Задание 1. Ноль или отсутствие данных

В наборе наблюдений `None` означает, что значение не опубликовано, а `0` — опубликованное нулевое значение. Исправьте фильтрацию так, чтобы она удаляла только пропуски. Объясните, почему проверка `if value` искажает данные.

In [ ]:
observations = [1.2, 0.0, None, -0.4, 0, None, 2.1]

# TODO: исправьте условие.
published = [value for value in observations if value]

assert published == [1.2, 0.0, -0.4, 0, 2.1]

## Задание 2. Копия сценария

Аналитик хочет изменить прогноз расходов, не затрагивая базовый бюджет. Код ниже создаёт новый словарь, но базовый сценарий всё равно меняется. Найдите общие объекты и создайте независимую копию без `copy.deepcopy`.

In [ ]:
base = {
    "revenue": [100, 110, 120],
    "costs": [70, 75, 80],
}

# TODO: замените поверхностное копирование осознанным копированием структуры.
scenario = base.copy()
scenario["costs"][1] = 95

assert base == {"revenue": [100, 110, 120], "costs": [70, 75, 80]}
assert scenario["costs"] == [70, 95, 80]
assert all(scenario[key] is not base[key] for key in base)

## Задание 3. Матрица прогнозов

Нужно создать независимые строки для трёх сценариев и четырёх кварталов. Исправьте инициализацию. После изменения одного квартала остальные строки не должны меняться.

In [ ]:
# TODO: исправьте создание матрицы.
forecast = [[0.0] * 4] * 3
forecast[0][2] = 1.5

assert forecast == [
    [0.0, 0.0, 1.5, 0.0],
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
]

## Денежные вычисления

Сумма большого числа операций должна совпадать с правилами учёта, а не быть «достаточно близкой». Сравним накопление в `float` и `Decimal`.

In [ ]:
from decimal import Decimal

float_total = sum([0.1] * 10)
decimal_total = sum([Decimal("0.1")] * 10, start=Decimal("0"))
print(float_total, decimal_total)

## Задание 4. Баланс по операциям

Вычислите точные остатки по счетам. Исходные суммы приходят строками, потому что именно так десятичные значения часто представлены в CSV и JSON.

In [ ]:
operations = [
    {"account": "cash", "amount": "100.10"},
    {"account": "deposit", "amount": "500.00"},
    {"account": "cash", "amount": "-0.20"},
    {"account": "cash", "amount": "0.10"},
]

balances = {}
# TODO: заполните balances, используя Decimal и словарь.

assert balances == {
    "cash": Decimal("100.00"),
    "deposit": Decimal("500.00"),
}

## Выбор контейнера

Для каждой задачи выберите основной контейнер и объясните, какое ограничение предметной области он выражает:

1. последовательность ежедневных цен;
2. уникальный набор валют в портфеле;
3. курс по коду валюты;
4. ключ наблюдения «показатель, год, квартал»;
5. строки исходного файла, среди которых могут быть дубликаты.

В пятом случае может понадобиться два контейнера: один хранит исходный порядок, другой помогает обнаруживать повторы.

In [ ]:
# TODO: замените None подходящими контейнерами и добавьте по два примера.
daily_prices = None
portfolio_currencies = None
rates_by_currency = None
observation_key = None

assert isinstance(daily_prices, list)
assert isinstance(portfolio_currencies, set)
assert isinstance(rates_by_currency, dict)
assert isinstance(observation_key, tuple)

## Задание 5. Мини-кейс: реестр наблюдений

Постройте словарь `registry`, где ключ — кортеж `(indicator, year, quarter)`, а значение — наблюдение. При повторении ключа должна сохраниться последняя опубликованная величина. Затем получите множество всех показателей и список ключей в порядке их первого появления в словаре.

In [ ]:
rows = [
    ("GDP", 2025, 1, 101.2),
    ("CPI", 2025, 1, 107.4),
    ("GDP", 2025, 2, 102.0),
    ("GDP", 2025, 1, 101.3),
]

registry = {}
# TODO: заполните registry.

indicators = set()  # TODO
keys_in_order = []  # TODO

assert registry[("GDP", 2025, 1)] == 101.3
assert indicators == {"GDP", "CPI"}
assert keys_in_order == [
    ("GDP", 2025, 1),
    ("CPI", 2025, 1),
    ("GDP", 2025, 2),
]

## Разбор решений

При обсуждении каждой задачи ответьте на три вопроса:

1. Какие объекты были созданы?
2. Какие имена ссылались на один объект?
3. Как выбранный контейнер выражает смысл данных?

Если объяснение опирается только на наблюдаемый вывод, попробуйте предсказать изменение программы до следующего запуска.

## Самопроверка

1. Как удалить только `None`, не потеряв числовые нули?
2. Почему `base.copy()` недостаточно для словаря списков?
3. Как создать матрицу с независимыми строками?
4. Почему денежные строки сначала преобразуют в `Decimal`, а не во `float`?
5. Какой контейнер одновременно удаляет повторы и почему он не всегда подходит для хранения исходных данных?
6. Может ли кортеж содержать список? Будет ли такой кортеж допустимым ключом словаря?

## Итоги

На семинаре модель «имя → объект» использована для диагностики реальных ошибок: потерянных нулей, общих вложенных списков и неточных денежных сумм. Выбор контейнера рассматривался как часть модели предметной области.

Далее решите [три задачи Easy и две Medium](tasks.md).